# Test the evaluation strategy

In [2]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 19.2 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 31.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install "google-cloud-aiplatform[evaluation]"

  Using cached ruamel_yaml-0.19.1-py3-none-any.whl.metadata (16 kB)
  Using cached litellm-1.85.7-py3-none-any.whl.metadata (33 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-macosx_11_0_arm64.whl.metadata (9.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.7 kB)
  Using cached hf_xet-1.5.2-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.24.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached litellm-1.85.7-py3-none-any.whl (17.0 MB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 9.7 MB/s  0:00:00
Using cached tokenizers-0.23.1-cp310-abi3-macosx_11_0_arm64.whl (3.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 21.9 MB/s  0:00:00
Using cached hf_xet-1.5.2-cp38-

In [12]:
pip install db-dtypes

Note: you may need to restart the kernel to use updated packages.


In [40]:
import pandas as pd
from google.cloud import bigquery
import vertexai
from vertexai.evaluation import EvalTask, MetricPromptTemplateExamples

# Configuration
PROJECT_ID = "life-coach-tracker"
LOCATION = "us-central1"
BQ_DATASET = "adk_agent_analytics"
EXPERIMENT_NAME = "agent-groundedness-eval"

# Initialize Google Cloud Clients
bq_client = bigquery.Client(project=PROJECT_ID)
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Step 1: BigQuery SQL Extraction
extraction_query = f"""
    SELECT
        TO_JSON_STRING(u.request_content) AS prompt,
        TO_JSON_STRING(a.response) AS response,
        -- Aggregate all tool outputs in the session to serve as grounding context
        STRING_AGG(TO_JSON_STRING(t.tool_result), ",") AS context
    FROM
        `{PROJECT_ID}.{BQ_DATASET}.v_llm_request` u
    JOIN
        `{PROJECT_ID}.{BQ_DATASET}.v_llm_response` a 
        ON u.invocation_id = a.invocation_id
    LEFT JOIN
        `{PROJECT_ID}.{BQ_DATASET}.v_tool_completed` t 
        ON u.invocation_id = t.invocation_id
    GROUP BY
        u.invocation_id,
        TO_JSON_STRING(u.request_content),
        TO_JSON_STRING(a.response)

    LIMIT 100
"""

print("Extracting trace data from BigQuery...")
eval_dataset_df = bq_client.query(extraction_query).to_dataframe()
eval_dataset_df = eval_dataset_df.dropna(subset=['context'])

# Create metric
from vertexai.evaluation import PointwiseMetric
from vertexai.evaluation.metrics import PointwiseMetricPromptTemplate

agent_groundedness_metric = PointwiseMetric(
    metric="agent_groundedness",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "agent_groundedness": (
                "Check whether every factual claim in the response is "
                "directly supported by the tool outputs in context. "
                "Claims requiring reasonable synthesis or reasoning across "
                "multiple tool outputs are acceptable. Claims not traceable "
                "to any tool output, or contradicting a tool output, are not."
            ),
        },
        rating_rubric={
            "5": "Every claim is fully traceable to tool outputs, with only reasonable synthesis.",
            "4": "Nearly all claims traceable; minor unsupported detail with low impact.",
            "3": "Mostly grounded but contains at least one claim not supported by context.",
            "2": "Multiple unsupported or contradicted claims.",
            "1": "Response is largely fabricated relative to the tool outputs provided.",
        },
        input_variables=["response", "context"],
    ),
)

# Step 2: Initialize and Run EvalTask
print("Initializing Vertex AI EvalTask...")
# Define the evaluation task with the Groundedness metric
groundedness_eval_task = EvalTask(
    dataset=eval_dataset_df,
    metrics=[agent_groundedness_metric],
    experiment=EXPERIMENT_NAME
)

print("Executing LLM-as-a-judge Groundedness Evaluation...")
eval_result = groundedness_eval_task.evaluate()

# Step 3: Output and Interpret Results
print("\n--- Evaluation Summary ---")
for metric, score in eval_result.summary_metrics.items():
    print(f"{metric}: {score}")

print("\n--- Detailed Row-by-Row Metrics ---")
detailed_metrics_df = eval_result.metrics_table
#for index, row in detailed_metrics_df.head(5).iterrows():
#    print(f"\nTrace #{index + 1}")
#    print(f"Prompt: {row['prompt'][:100]}...")
    #print(f"Groundedness Score: {row['groundedness/score']}")
    #print(f"Judge Explanation: {row['groundedness/explanation']}")

Extracting trace data from BigQuery...
Initializing Vertex AI EvalTask...
Executing LLM-as-a-judge Groundedness Evaluation...


Associating projects/338118379899/locations/us-central1/metadataStores/default/contexts/agent-groundedness-eval-ebf481c2-1a9f-46b8-a25a-1109978236e3 to Experiment: agent-groundedness-eval


Computing metrics with a total of 49 Vertex Gen AI Evaluation Service API requests.


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 49 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 49/49 [00:23<00:00,  2.06it/s]

All 49 metric requests are successfully computed.



INFO:vertexai.evaluation._evaluation:All 49 metric requests are successfully computed.


Evaluation Took:23.770796291995794 seconds


INFO:vertexai.evaluation._evaluation:Evaluation Took:23.770796291995794 seconds



--- Evaluation Summary ---
row_count: 49
agent_groundedness/mean: 4.1020408163265305
agent_groundedness/std: 1.1038738220312982

--- Detailed Row-by-Row Metrics ---


In [41]:
eval_dataset_df

,prompt,response,context
0,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, the first task, \""Define Agent E...","{""complexity"":""medium"",""effort_points"":5,""esti..."
1,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Design Agent Evaluation Methodology ...","{""complexity"":""medium"",""effort_points"":5,""esti..."
2,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Define Core Evaluation Metrics and C...","{""complexity"":""medium"",""effort_points"":5,""esti..."
3,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Develop/Acquire Evaluation Tools and...","{""complexity"":""medium"",""effort_points"":5,""esti..."
4,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'The task \""Identify and Document Solut...","{""complexity"":""medium"",""effort_points"":5,""esti..."
5,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, I have estimated all the initial...","{""complexity"":""medium"",""effort_points"":5,""esti..."
6,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, I can help you plan this project...","{""complexity"":""medium"",""effort_points"":5,""esti..."
7,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, the first task, \""Define Agent E...","{""complexity"":""medium"",""effort_points"":5,""esti..."
8,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Design Agent Evaluation Methodology ...","{""complexity"":""medium"",""effort_points"":5,""esti..."
9,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Define Core Evaluation Metrics and C...","{""complexity"":""medium"",""effort_points"":5,""esti..."


In [42]:
detailed_metrics_df[["prompt", "response",]]

,prompt,response
0,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, the first task, \""Define Agent E..."
1,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Design Agent Evaluation Methodology ..."
2,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Define Core Evaluation Metrics and C..."
3,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Develop/Acquire Evaluation Tools and..."
4,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'The task \""Identify and Document Solut..."
5,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, I have estimated all the initial..."
6,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, I can help you plan this project..."
7,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, the first task, \""Define Agent E..."
8,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Design Agent Evaluation Methodology ..."
9,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Define Core Evaluation Metrics and C..."


In [ ]:
print(detailed_metrics_df["prompt"].iloc[3])

{"prompt":[{"content":"Hi","role":"user"},{"content":"Hello! I'm your AI project manager. How can I help you today?\n","role":"model"},{"content":"Help me plan a project to define and create agent evaluations for multiple solutions. This project should create the strategy ","role":"user"},{"content":"Okay, I can help you plan this project. To start, let's break down the goal into some initial tasks. This project will involve defining a strategy, identifying solutions, and then designing and creating the evaluation mechanisms.\n\nHere's a proposed set of initial tasks. I'll estimate each one for you:\n\n1.  **Define Agent Evaluation Strategy:** This task involves outlining the overall approach, goals, and principles for evaluating agents across different solutions.\n2.  **Identify and Document Solutions for Evaluation:** This task focuses on listing all the solutions that need agent evaluations and understanding their specific characteristics relevant to evaluation.\n3.  **Define Core E

In [ ]:
print(detailed_metrics_df["context"].iloc[3])

{"complexity":"medium","effort_points":5,"estimated_hours":16,"task_name":"Identify and Document Solutions for Evaluation"},{"complexity":"high","effort_points":13,"estimated_hours":40,"task_name":"Define Core Evaluation Metrics and Criteria"},{"complexity":"high","effort_points":13,"estimated_hours":40,"task_name":"Develop/Acquire Evaluation Tools and Infrastructure"},{"complexity":"high","effort_points":13,"estimated_hours":40,"task_name":"Define Agent Evaluation Strategy"},{"complexity":"high","effort_points":13,"estimated_hours":40,"task_name":"Create Initial Evaluation Test Cases/Scenarios"},{"complexity":"high","effort_points":13,"estimated_hours":40,"task_name":"Design Agent Evaluation Methodology and Process"}


In [ ]:
print(detailed_metrics_df["response"].iloc[3])

"text: '\"Design Agent Evaluation Methodology and Process\" is estimated at **40 hours** and **13 effort points**, with high complexity.\n\nNext, let's estimate \"Develop/Acquire Evaluation Tools and Infrastructure.\" This task could involve significant development or integration effort.\n' | call: estimate_task"


In [51]:
print(detailed_metrics_df["agent_groundedness/explanation"].iloc[3])

The first part of the response, detailing 'Develop/Acquire Evaluation Tools and Infrastructure', is fully grounded in the provided context. The second part correctly identifies the task 'Create Initial Evaluation Test Cases/Scenarios', but the accompanying statement 'This will be crucial for validating the evaluation framework' is an interpretive claim not directly supported by the context. However, it's a minor unsupported detail with low impact, leading to a score of 4.


In [52]:
detailed_metrics_df

,prompt,response,context,agent_groundedness/explanation,agent_groundedness/score
0,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, the first task, \""Define Agent E...","{""complexity"":""medium"",""effort_points"":5,""esti...","Every factual claim in the response, specifica...",5.0
1,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Design Agent Evaluation Methodology ...","{""complexity"":""medium"",""effort_points"":5,""esti...",Every factual claim made in the response is di...,5.0
2,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Define Core Evaluation Metrics and C...","{""complexity"":""medium"",""effort_points"":5,""esti...",Every factual claim in the response is directl...,5.0
3,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Develop/Acquire Evaluation Tools and...","{""complexity"":""medium"",""effort_points"":5,""esti...","The first part of the response, detailing 'Dev...",4.0
4,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'The task \""Identify and Document Solut...","{""complexity"":""medium"",""effort_points"":5,""esti...","Every factual claim in the response, including...",5.0
5,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, I have estimated all the initial...","{""complexity"":""medium"",""effort_points"":5,""esti...",The AI response correctly extracts all task na...,2.0
6,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, I can help you plan this project...","{""complexity"":""medium"",""effort_points"":5,""esti...",Every task listed in the response is directly ...,5.0
7,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: 'Okay, the first task, \""Define Agent E...","{""complexity"":""medium"",""effort_points"":5,""esti...",Every factual claim made in the response regar...,5.0
8,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Design Agent Evaluation Methodology ...","{""complexity"":""medium"",""effort_points"":5,""esti...",All factual claims made in the response regard...,5.0
9,"{""prompt"":[{""content"":""Hi"",""role"":""user""},{""co...","""text: '\""Define Core Evaluation Metrics and C...","{""complexity"":""medium"",""effort_points"":5,""esti...",The response accurately extracts and presents ...,3.0


In [53]:
detailed_metrics_df["agent_groundedness/score"].mean()

np.float64(4.1020408163265305)